# Fine-tuning Gemma 2 2B for Tool Use (QLoRA)

Fine-tunes `google/gemma-2-2b-it` on the Salesforce xLAM-60k function-calling dataset using QLoRA on a **free Colab T4 GPU**.

**Why Gemma 2 2B?** Text-only, genuinely 2B params, trains in ~1-2 hours on a free T4. Same `<start_of_turn>` chat template as Gemma 4 so the dataset format is identical.

**Before running anything:**
1. Runtime → Change runtime type → **T4 GPU**
2. Click the 🔑 icon in the left sidebar → add these 4 secrets with "Notebook access" ON:
   - `HF_TOKEN` — huggingface.co → Settings → Access Tokens → New token (write)
   - `WANDB_API_KEY` — wandb.ai → Settings → API keys
   - `GROQ_API_KEY` — console.groq.com → API Keys (free)
   - `GEMINI_API_KEY` — aistudio.google.com → Get API key (free)

**Then run cells top to bottom with Shift+Enter. Do not skip any cell.**

## Step 1 — Check GPU

Verify Colab gave us a T4. If you see "No GPU", go to Runtime → Change runtime type → T4 GPU.

In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'✓ GPU found: {result.stdout.strip()}')
else:
    print('✗ No GPU — go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Install dependencies

Installs transformers from GitHub source (safest for latest model support), plus PEFT, TRL, bitsandbytes, and other training libraries. Takes ~3 minutes.

In [ ]:
%%capture
# transformers must come from GitHub — stable pip release doesn't support Gemma 4 yet
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q \
    peft>=0.12.0 \
    trl>=0.9.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.31.0 \
    datasets>=2.19.0 \
    wandb>=0.17.0 \
    huggingface-hub>=0.23.0 \
    sentencepiece>=0.2.0 \
    protobuf>=5.27.0 \
    python-dotenv>=1.0.0 \
    pyyaml>=6.0

import transformers
print(f'✓ transformers {transformers.__version__}')
print('✓ All dependencies installed')

## Step 3 — Load secrets

Reads your API keys from the Colab Secrets vault (🔑 sidebar). Keys never appear in this file.

In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY']  = userdata.get('WANDB_API_KEY')
os.environ['GROQ_API_KEY']   = userdata.get('GROQ_API_KEY')
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

checks = {
    'HF_TOKEN':       lambda v: v and v.startswith('hf_') and len(v) > 20,
    'WANDB_API_KEY':  lambda v: bool(v),
    'GROQ_API_KEY':   lambda v: bool(v),
    'GEMINI_API_KEY': lambda v: bool(v),
}
for key, ok in checks.items():
    status = '✓' if ok(os.environ[key]) else '✗ MISSING — check 🔑 sidebar'
    print(f'{status} {key}')

## Step 4 — Clone repo and prepare data

Clones your GitHub repo, then runs `data/prepare.py` to download and format the xLAM-60k dataset. Takes ~10 minutes (mostly the dataset download). Skips automatically if data already exists.

In [ ]:
import os
from pathlib import Path

REPO = 'function-call-finetune'
REPO_URL = 'https://github.com/sahilmdeshmukh/function-call-finetune.git'

if Path(REPO).exists():
    print('Repo exists — pulling latest fixes...')
    !git -C {REPO} pull
else:
    print('Cloning repo...')
    !git clone {REPO_URL}

os.chdir(REPO)
print(f'✓ Working directory: {os.getcwd()}')

# Prepare dataset (skips if already done)
if Path('data/train.jsonl').exists():
    print('✓ Data already prepared')
    !echo "  train: $(wc -l < data/train.jsonl) examples"
    !echo "  val:   $(wc -l < data/val.jsonl) examples"
    !echo "  test:  $(wc -l < data/test_held_out.jsonl) examples"
else:
    print('Preparing dataset (~10 min)...')
    !python data/prepare.py

## Step 5 — Sanity check the data format

Print 2 examples and verify the `<start_of_turn>user` / `<start_of_turn>model` structure looks right before training. If it looks wrong, stop here and fix it — don't waste 90 minutes training on bad data.

In [ ]:
!python data/inspect.py --n 2

## Step 6 — Train

Loads Gemma 2 2B in 4-bit (~1GB), attaches LoRA adapters, and trains for 1 epoch (~1-2 hours on T4).

**What you'll see:**
- `trainable params: ~24M || all params: ~2B || trainable%: ~1.1%` — LoRA working
- Loss numbers every 25 steps — should drop from ~1.5 down toward ~0.4
- Checkpoints saved every 200 steps — safe against disconnects

**If Colab disconnects:** re-run from Cell 1. Training resumes from the last checkpoint automatically.

In [ ]:
import gc, torch

# Clear any leftover GPU memory before loading the model
gc.collect()
torch.cuda.empty_cache()
print(f'Free GPU memory: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

!python train.py --config configs/train.yaml

## Step 7 — Verify adapter is on Hugging Face Hub

After training finishes, confirms the adapter was pushed successfully.

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi(token=os.environ['HF_TOKEN'])
repo_id = 'sahilmdeshmukh/gemma-2-2b-tool-use-lora'

try:
    files = list(api.list_repo_files(repo_id))
    print(f'✓ Adapter live at: huggingface.co/{repo_id}')
    print(f'  Files: {files}')
except Exception as e:
    print(f'✗ Repo not found: {e}')

## Step 8 — Quick inference test

Load the fine-tuned adapter and run one sample tool call to confirm it works.

In [ ]:
import json, torch, os, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()

BASE    = 'google/gemma-2-2b-it'
ADAPTER = 'sahilmdeshmukh/gemma-2-2b-tool-use-lora'
TOKEN   = os.environ['HF_TOKEN']

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(BASE, token=TOKEN)
base  = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                              device_map='auto', token=TOKEN)
model = PeftModel.from_pretrained(base, ADAPTER, token=TOKEN)
model.eval()

tools = [{"name": "get_weather", "description": "Get current weather",
           "parameters": {"type": "object",
                          "properties": {"location": {"type": "string"},
                                         "unit": {"type": "string"}},
                          "required": ["location"]}}]

system = ("You have access to the following functions:\n\n"
          + json.dumps(tools, indent=2)
          + '\n\nRespond with: {"name": "...", "arguments": {...}}')

messages = [{"role": "user", "content": f"{system}\n\nWhat is the weather in Mumbai in celsius?"}]
inputs = tokenizer.apply_chat_template(messages, return_tensors='pt',
                                       add_generation_prompt=True).to(model.device)

with torch.no_grad():
    out = model.generate(inputs, max_new_tokens=128, temperature=0.1, do_sample=True)

response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print('Output:', response)

try:
    parsed = json.loads(response.strip())
    print(f'\n✓ Valid JSON — function: {parsed["name"]}, args: {parsed["arguments"]}')
except:
    print('\n✗ Not valid JSON — may need more training')